# Bloque 4 — Agentes de IA
## Notebook 02: Agentes con herramientas (tool-use)

---

### El problema del notebook anterior

En el notebook 01, incrustamos todos los datos directamente en el texto de las tareas. Esto funciona para datasets pequeños, pero tiene dos limitaciones:

1. **Escala**: si hay 10.000 actores en vez de 30, no cabe en el contexto del LLM.
2. **Precisión**: el modelo puede alucindar al parafrasear los datos. Queremos que los números vengan de pandas, no de la imaginación del LLM.

### La solución: herramientas

Una **herramienta** es una función Python que el agente puede invocar cuando necesita información específica. El LLM decide **cuándo** llamarla y **con qué argumentos**.

```
┌─────────────────────────────────────────────────────────┐
│                        AGENTE                           │
│                                                         │
│  Pregunta: "¿Quién domina la categoría 'financial'?"    │
│                                                         │
│  Razona: "Necesito consultar la distribución...         │
│           voy a llamar top_actores_por_categoria()"     │
│                                                         │
│  Llama herramienta ─────────────────────────────────▶  │
│               [top_actores_por_categoria('financial')]  │
│  Recibe resultado ◀─────────────────────────────────── │
│               ["stern: 12", "mango: 8", ...]           │
│                                                         │
│  Responde con datos reales (no alucinados)              │
└─────────────────────────────────────────────────────────┘
```

### El secreto del tool-use: el docstring

El LLM decide qué herramienta usar leyendo **su docstring**. Si el docstring es vago, el modelo elegirá mal. Si es preciso, elegirá bien.

```python
@tool
def mi_herramienta(arg: str) -> str:
    """Descripción precisa de CUÁNDO usar esta herramienta
    y QUÉ parámetros espera. El LLM lee esto para decidir."""
    ...
```

### Modelo elegido para este notebook

Usamos **`qwen2.5:14b`** (el analista) para el agente de tool-use. Tool-calling requiere que el modelo forme argumentos estructurados correctamente — qwen2.5 es más fiable en esto que deepseek-r1 para modelos de 14B.

Al final del notebook añadimos un segundo agente con **`deepseek-r1:14b`** que sintetiza los hallazgos — así mantenemos la distribución de modelos del bloque.

In [1]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os
import json
import concurrent.futures
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

os.environ['OPENAI_API_KEY'] = 'NA'

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool


def kickoff(crew, timeout=600):
    """Ejecuta crew.kickoff() en un hilo separado para evitar conflictos
    con el event loop de Jupyter/nbconvert (problema de CrewAI 1.x)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result(timeout=timeout)


# ─── RUTAS ───────────────────────────────────────────────────────────────────
DATA = Path('..') / 'bloque5_ransomware' / 'data_Vruto' / 'ContiLeaks'

PROFILES_JSON   = DATA / 'actor_profiles.json'
CLASSIFIED_PARQ = DATA / 'conti_sample_classified.parquet'
EMBEDDINGS_NPY  = DATA / 'message_embeddings.npy'

for p in [PROFILES_JSON, CLASSIFIED_PARQ, EMBEDDINGS_NPY]:
    assert p.exists(), f'No encontrado: {p}'

print('Rutas verificadas.')

Rutas verificadas.


In [2]:
# ─── CARGAR Y PREPARAR DATOS ──────────────────────────────────────────────────
# Cargamos todos los datos UNA SOLA VEZ al inicio.
# Las herramientas accederán a estas variables globales sin recargar los ficheros.

with open(PROFILES_JSON) as f:
    PROFILES = json.load(f)

DF = pd.read_parquet(CLASSIFIED_PARQ)

# Cargamos los embeddings (~2520 × 4096): un vector por cada mensaje clasificado.
EMBEDDINGS = np.load(EMBEDDINGS_NPY)

# Calculamos el centroide (vector promedio) de cada actor.
# El centroide representa el "estilo semántico" medio de ese actor:
# si mango habla mucho de negociaciones, su centroide estará cerca de otros
# mensajes de negociación en el espacio vectorial de 4096 dimensiones.
ACTOR_CENTROIDS = {}
for actor, grupo in DF.groupby('username'):
    indices = grupo.index.tolist()
    centroide = EMBEDDINGS[indices].mean(axis=0)
    norma = np.linalg.norm(centroide)
    ACTOR_CENTROIDS[actor] = centroide / norma if norma > 0 else centroide

print('Datos cargados:')
print(f'  Actores con perfil    : {len(PROFILES)}')
print(f'  Mensajes clasificados : {len(DF)}')
print(f'  Shape embeddings      : {EMBEDDINGS.shape}')
print(f'  Centroides calculados : {len(ACTOR_CENTROIDS)}')

Datos cargados:
  Actores con perfil    : 30
  Mensajes clasificados : 2520
  Shape embeddings      : (2520, 4096)
  Centroides calculados : 30


## Las tres herramientas

Vamos a definir tres herramientas con responsabilidades distintas:

| Herramienta | Cuándo usarla | Qué devuelve |
|---|---|---|
| `buscar_actor` | Cuando necesitas el perfil de un actor concreto | Rol, confianza, resumen, evidencias |
| `top_actores_por_categoria` | Cuando necesitas saber quién domina una categoría | Lista de actores con conteos |
| `calcular_similitud` | Cuando necesitas comparar dos actores semánticamente | Similitud coseno 0–1 con interpretación |

El **docstring** de cada herramienta es lo que el LLM leerá para decidir cuándo invocarla. Cuanto más preciso sea, mejor tomará la decisión el modelo.

In [3]:
# ─── HERRAMIENTA 1: buscar_actor ──────────────────────────────────────────────

@tool
def buscar_actor(nombre: str) -> str:
    """Busca y devuelve el perfil completo de un actor del grupo Conti por su nombre.
    Usar cuando se necesite información sobre el rol, nivel de confianza o actividad
    de un actor específico. El nombre debe estar en minúsculas (ej: 'stern', 'mango')."""

    nombre = nombre.lower().strip()

    if nombre not in PROFILES:
        # Si el actor no existe, devolvemos los disponibles para que el agente
        # pueda corregir el nombre en la siguiente iteración.
        disponibles = ', '.join(sorted(PROFILES.keys()))
        return f'Actor "{nombre}" no encontrado. Actores disponibles: {disponibles}'

    info = PROFILES[nombre]
    evidencias = '\n  '.join(info['evidence'][:3])  # máximo 3 ejemplos

    return (
        f'Actor: {nombre}\n'
        f'Rol: {info["role"]}\n'
        f'Confianza: {info["confidence"]}\n'
        f'Resumen: {info["summary"]}\n'
        f'Evidencias (hasta 3):\n  {evidencias}'
    )


# ─── HERRAMIENTA 2: top_actores_por_categoria ─────────────────────────────────

@tool
def top_actores_por_categoria(categoria: str, n: int = 5) -> str:
    """Devuelve los N actores de Conti con más mensajes en una categoría dada.
    Categorías válidas: 'technical', 'comms', 'operational', 'financial',
    'organizational', 'unknown'. Usar cuando se quiera saber qué actores
    dominan un tipo de actividad concreto."""

    categorias_validas = sorted(DF['category'].unique())
    categoria = categoria.lower().strip()

    if categoria not in categorias_validas:
        return (
            f'Categoría "{categoria}" no válida. '
            f'Opciones: {categorias_validas}'
        )

    conteos = (
        DF[DF['category'] == categoria]
        .groupby('username')
        .size()
        .sort_values(ascending=False)
        .head(n)
    )

    if conteos.empty:
        return f'No hay mensajes en la categoría "{categoria}".'

    lineas = [f'{actor}: {count} mensajes' for actor, count in conteos.items()]
    return f'Top {n} actores en categoría "{categoria}":\n' + '\n'.join(lineas)


# ─── HERRAMIENTA 3: calcular_similitud ───────────────────────────────────────

@tool
def calcular_similitud(actor1: str, actor2: str) -> str:
    """Calcula la similitud semántica (coseno) entre dos actores de Conti
    basándose en el estilo y contenido de sus mensajes (embeddings vectoriales).
    Un valor de 1.0 indica estilos idénticos; 0.0 indica estilos completamente distintos.
    Usar para comparar dos actores concretos o para encontrar quién es más similar a alguien."""

    actor1 = actor1.lower().strip()
    actor2 = actor2.lower().strip()

    actores_disponibles = list(ACTOR_CENTROIDS.keys())

    if actor1 not in ACTOR_CENTROIDS:
        return f'Actor "{actor1}" no encontrado. Disponibles: {actores_disponibles}'
    if actor2 not in ACTOR_CENTROIDS:
        return f'Actor "{actor2}" no encontrado. Disponibles: {actores_disponibles}'

    # cosine_similarity espera matrices 2D, por eso hacemos reshape a (1, N).
    v1 = ACTOR_CENTROIDS[actor1].reshape(1, -1)
    v2 = ACTOR_CENTROIDS[actor2].reshape(1, -1)
    sim = float(cosine_similarity(v1, v2)[0][0])

    if sim > 0.95:
        interpretacion = 'muy alta — estilos de comunicación muy similares'
    elif sim > 0.85:
        interpretacion = 'alta — patrones de comunicación parecidos'
    elif sim > 0.70:
        interpretacion = 'moderada — cierta similitud en el estilo'
    else:
        interpretacion = 'baja — estilos de comunicación distintos'

    return (
        f'Similitud entre "{actor1}" y "{actor2}": {sim:.4f}\n'
        f'Interpretación: {interpretacion}'
    )


print('Tres herramientas definidas: buscar_actor, top_actores_por_categoria, calcular_similitud')

Tres herramientas definidas: buscar_actor, top_actores_por_categoria, calcular_similitud


## El agente analista con herramientas

Creamos un agente con `qwen2.5:14b` que tiene acceso a las tres herramientas. El LLM decidirá cuándo y cuál usar basándose en la pregunta que le hacemos.

Usamos `qwen2.5:14b` aquí porque tool-calling requiere que el modelo forme argumentos estructurados correctamente (nombres en minúsculas, tipos correctos). En pruebas locales, `qwen2.5:14b` es más consistente que `deepseek-r1:14b` para este patrón específico.

In [4]:
# ─── CONFIGURACIÓN DE LLMs ────────────────────────────────────────────────────
OLLAMA_URL = 'http://localhost:11434'

# qwen2.5:14b para tool-use — mejor en formación de argumentos estructurados
llm_analista = LLM(
    model='ollama/qwen2.5:14b',
    base_url=OLLAMA_URL,
)

# deepseek-r1:14b para síntesis final — razona con CoT antes de escribir
llm_redactor = LLM(
    model='ollama/deepseek-r1:14b',
    base_url=OLLAMA_URL,
)

# ─── AGENTE CON HERRAMIENTAS (qwen2.5:14b) ────────────────────────────────────
# La lista `tools` le dice al agente qué herramientas tiene disponibles.
# El backstory menciona explícitamente que debe usar las herramientas en vez
# de inventarse los datos — esto reduce las alucinaciones notablemente.
agente_analista = Agent(
    role='Investigador analítico de grupos de ransomware',
    goal=(
        'Responder preguntas sobre el grupo Conti con datos precisos, '
        'usando las herramientas disponibles para consultar los datos reales.'
    ),
    backstory=(
        'Eres un investigador de CTI que trabaja con una base de datos de actores de Conti. '
        'Tienes acceso a tres herramientas de consulta: buscar_actor, top_actores_por_categoria '
        'y calcular_similitud. SIEMPRE usas estas herramientas para obtener datos precisos '
        'en vez de recordar información de tu entrenamiento. '
        'Si necesitas el perfil de un actor, llamas a buscar_actor. '
        'Si necesitas rankings por actividad, llamas a top_actores_por_categoria. '
        'Si necesitas comparar dos actores, llamas a calcular_similitud. '
        'Respondes en español de forma concisa y factual.'
    ),
    tools=[buscar_actor, top_actores_por_categoria, calcular_similitud],
    llm=llm_analista,
    verbose=True,
)

# ─── AGENTE REDACTOR FINAL (deepseek-r1:14b) ──────────────────────────────────
# Recibe los hallazgos del analista y los sintetiza en un resumen ejecutivo.
# No tiene herramientas — su única función es razonar y escribir bien.
agente_redactor = Agent(
    role='Redactor de síntesis de Threat Intelligence',
    goal=(
        'Producir un resumen ejecutivo conciso con los hallazgos '
        'del análisis de herramientas sobre el grupo Conti.'
    ),
    backstory=(
        'Eres un redactor técnico especializado en CTI. '
        'Recibes los hallazgos de un analista y los conviertes en un resumen ejecutivo '
        'claro, estructurado y accionable. '
        'Nunca inventas datos — solo sintetizas lo que el analista ha encontrado. '
        'Escribes en español.'
    ),
    llm=llm_redactor,
    verbose=True,
)

print('Agentes configurados:')
print(f'  agente_analista  → {llm_analista.model} + 3 herramientas')
print(f'  agente_redactor  → {llm_redactor.model} (sin herramientas, solo síntesis)')

Agentes configurados:
  agente_analista  → qwen2.5:14b + 3 herramientas
  agente_redactor  → deepseek-r1:14b (sin herramientas, solo síntesis)


In [5]:
# ─── FUNCIÓN AUXILIAR ─────────────────────────────────────────────────────────
# Creamos una función que lanza el crew con una investigación y devuelve
# tanto el análisis como la síntesis final.

def investigar(pregunta: str) -> str:
    """Lanza un crew analista+redactor con una pregunta y devuelve la síntesis."""

    tarea_analisis = Task(
        description=pregunta,
        expected_output=(
            'Respuesta factual basada en los datos de las herramientas. '
            'Cita explícitamente los valores numéricos y nombres obtenidos.'
        ),
        agent=agente_analista,
    )

    tarea_sintesis = Task(
        description=(
            'Redacta un resumen ejecutivo de 2-3 párrafos con los hallazgos '
            'del análisis anterior. Destaca las implicaciones para equipos de defensa.'
        ),
        expected_output='Resumen ejecutivo en español, máximo 200 palabras.',
        agent=agente_redactor,
        context=[tarea_analisis],
    )

    crew = Crew(
        agents=[agente_analista, agente_redactor],
        tasks=[tarea_analisis, tarea_sintesis],
        process=Process.sequential,
        verbose=True,
    )
    return kickoff(crew).raw

print('Función investigar() lista.')

Función investigar() lista.


## Cinco preguntas de dificultad creciente

Observa cómo el agente decide qué herramienta usar para responder cada pregunta. La dificultad aumenta porque las últimas preguntas requieren encadenar varias herramientas.

In [6]:
# ─── PREGUNTA 1 (simple) — perfil de un actor ─────────────────────────────────
# Herramienta esperada: buscar_actor('stern')
# El agente debería llamar directamente a buscar_actor con el nombre correcto.

p1 = '¿Qué rol tiene el actor "stern" en el grupo Conti y cuál es su nivel de confianza?'
print(f'PREGUNTA 1: {p1}')
print('─' * 60)
r1 = investigar(p1)
print('\nSÍNTESIS FINAL:')
print(r1)

PREGUNTA 1: ¿Qué rol tiene el actor "stern" en el grupo Conti y cuál es su nivel de confianza?
────────────────────────────────────────────────────────────


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6c05d969-80d6-4bdc-bcce-90cd337a95e4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: ¿Qué rol tiene el actor "stern" en el grupo Conti y cuál es su nivel de confianza?                       │
│  ID: 471eeab8-90c3-4d08-9803-6305397ec829                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador analítico de grupos de ransomware                                                          │
│                                                                                                                 │
│  Task: ¿Qué rol tiene el actor "stern" en el grupo Conti y cuál es su nivel de confianza?                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Resumen Ejecutivo: Análisis del Grupamento Conti y su Miembro "stern"**                                      │
│                                                                                                                 │
│  El análisis reciente sobre el grupo Conti revela que uno de sus miembros, identificado como "stern," ocupa el  │
│  rol de **operador** dentro de la organización. Este individuo se dedica a gestionar tareas relacionadas con    │
│  el despliegue de ransomware, lo que incluye probables actividades de comunicación con otros miembros del       │
│  equipo o afiliados externos. Además, "stern" coordina funciones tácticas como el testing de herramientas       │
│  nuevas, otorgamiento de permisos de acceso y mantenimiento del funcionamiento óptimo de los sistemas.          │
│                                                                                                                 │
│  Entre las evidencias analizadas destaca una serie de mensajes en ruso que sugieren preguntas sobre la          │
│  efectividad de un generador o testificador de ransomware. Por ejemplo:                                         │
│  1. "Привет как там с инкубатором генератором?" ( traducción literal: "Hola, ¿cómo está el incubador del        │
│  generador?").                                                                                                  │
│  2. "а кто тестировал его?" (traducción literal: "¿Quién lo probó?").                                           │
│  3. "возьми у профа семпл локера maze" (traducción literal: "toma una muestra del locker Maze del profesor").   │
│                                                                                                                 │
│  Estas evidencias refuerzan la hipótesis de que "stern" está involucrado en pruebas y validaciones de           │
│  herramientas criminales, lo cual es crítico para sustraer el ransomware antes de su uso operativo. La          │
│  información obtenida permite a los equipos de defensa comprender mejor las dinámicas internas del grupo Conti  │
│  y anticipar posibles campañas futuras.                                                                         │
│                                                                                                                 │
│  **Implicaciones para los Equipos de Defensa:**                                                                 │
│  El conocimiento detallado sobre el rol y actividades de "stern" permite a las organizaciones mejorar sus       │
│  defensas contra ransomware, detectando y bloqueando actividades sospechas tempranamente. Además, la            │
│  identificación de posibles afiliados o herramientas asociadas al grupo facilita una respuesta más estratégica  │
│  y efectiva contra sus amenazas.                                                                                │
│                                                                                                                 │
│  En conclusión, el análisis del miembro "stern" del grupo Conti subraya la importancia de mantener un enfoque   │
│  proactivo en la detección y mitigación de actividades ransomware-related, basado en inteligencia colectiva e   │
│  información confiable.                                                                                         │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un resumen ejecutivo de 2-3 párrafos con los hallazgos del análisis anterior. Destaca las        │
│  implicaciones para equipos de defensa.                                                                         │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6c05d969-80d6-4bdc-bcce-90cd337a95e4                                                                       │
│  Final Output: **Resumen Ejecutivo: Análisis del Grupamento Conti y su Miembro "stern"**                        │
│                                                                                                                 │
│  El análisis reciente sobre el grupo Conti revela que uno de sus miembros, identificado como "stern," ocupa el  │
│  rol de **operador** dentro de la organización. Este individuo se dedica a gestionar tareas relacionadas con    │
│  el despliegue de ransomware, lo que incluye probables actividades de comunicación con otros miembros del       │
│  equipo o afiliados externos. Además, "stern" coordina funciones tácticas como el testing de herramientas       │
│  nuevas, otorgamiento de permisos de acceso y mantenimiento del funcionamiento óptimo de los sistemas.          │
│                                                                                                                 │
│  Entre las evidencias analizadas destaca una serie de mensajes en ruso que sugieren preguntas sobre la          │
│  efectividad de un generador o testificador de ransomware. Por ejemplo:                                         │
│  1. "Привет как там с инкубатором генератором?" ( traducción literal: "Hola, ¿cómo está el incubador del        │
│  generador?").                                                                                                  │
│  2. "а кто тестировал его?" (traducción literal: "¿Quién lo probó?").                                           │
│  3. "возьми у профа семпл локера maze" (traducción literal: "toma una muestra del locker Maze del profesor").   │
│                                                                                                                 │
│  Estas evidencias refuerzan la hipótesis de que "stern" está involucrado en pruebas y validaciones de           │
│  herramientas criminales, lo cual es crítico para sustraer el ransomware antes de su uso operativo. La          │
│  información obtenida permite a los equipos de defensa comprender mejor las dinámicas internas del grupo Conti  │
│  y anticipar posibles campañas futuras.                                                                         │
│                                                                                                                 │
│  **Implicaciones para los Equipos de Defensa:**                                                                 │
│  El conocimiento detallado sobre el rol y actividades de "stern" permite a las organizaciones mejorar sus       │
│  defensas contra ransomware, detectando y bloqueando actividades sospechas tempranamente. Además, la            │
│  identificación de posibles afiliados o herramientas asociadas al grupo facilita una respuesta más estratégica  │
│  y efectiva contra sus amenazas.                                                                                │
│                                                                                                                 │
│  En conclusión, el análisis del miembro "stern" del grupo Conti subraya la importancia de mantener un enfoque   │
│  proactivo en la detección y mitigación de actividades ransomware-related, basado en inteligencia colectiva e   │
│  información confiable.                                                                                         │
│                                                       


SÍNTESIS FINAL:
**Resumen Ejecutivo: Análisis del Grupamento Conti y su Miembro "stern"**

El análisis reciente sobre el grupo Conti revela que uno de sus miembros, identificado como "stern," ocupa el rol de **operador** dentro de la organización. Este individuo se dedica a gestionar tareas relacionadas con el despliegue de ransomware, lo que incluye probables actividades de comunicación con otros miembros del equipo o afiliados externos. Además, "stern" coordina funciones tácticas como el testing de herramientas nuevas, otorgamiento de permisos de acceso y mantenimiento del funcionamiento óptimo de los sistemas.

Entre las evidencias analizadas destaca una serie de mensajes en ruso que sugieren preguntas sobre la efectividad de un generador o testificador de ransomware. Por ejemplo:  
1. "Привет как там с инкубатором генератором?" ( traducción literal: "Hola, ¿cómo está el incubador del generador?").  
2. "а кто тестировал его?" (traducción literal: "¿Quién lo probó?").  
3. "возьми 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# ─── PREGUNTA 2 (simple) — ranking por categoría ──────────────────────────────
# Herramienta esperada: top_actores_por_categoria('financial', 3)

p2 = '¿Qué tres actores de Conti tienen más mensajes de categoría financiera?'
print(f'PREGUNTA 2: {p2}')
print('─' * 60)
r2 = investigar(p2)
print('\nSÍNTESIS FINAL:')
print(r2)

PREGUNTA 2: ¿Qué tres actores de Conti tienen más mensajes de categoría financiera?
────────────────────────────────────────────────────────────


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6d2d9a0e-8564-4a5f-9927-141b4554c1d5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: ¿Qué tres actores de Conti tienen más mensajes de categoría financiera?                                  │
│  ID: 5d7eaec9-9139-4e5d-85d6-44dbd6069e6b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador analítico de grupos de ransomware                                                          │
│                                                                                                                 │
│  Task: ¿Qué tres actores de Conti tienen más mensajes de categoría financiera?                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Resumen Ejecutivo: Análisis del Grupo Conti**                                                                │
│                                                                                                                 │
│  El grupo Conti, conocido por su actividad cibernética y su enfoque en amenazas financieras, ha demostrado una  │
│  intensa actividad reciente. Entre los principales actores de Conti con mensajes relacionados a la categoría    │
│  financiera, se destaca **strix** con 13 mensajes, seguido de **mango** con 11 y **bio** con 8 mensajes. Estos  │
│  tres actores parecen estar centrados en tareas operativas que involucran ataque directo contra entidades       │
│  financieras.                                                                                                   │
│                                                                                                                 │
│  Estos datos reflejan un posible aumento en las actividades del grupo Conti, y sugieren una creación de         │
│  contenido sistemático relacionado con la explotación financiera. Los equipos de defensa deben considerar a     │
│  estos actores prioritarios y monitorear su actividad, además de implementar estrategias para mitigar riesgos   │
│  en el sector bancario.                                                                                         │
│                                                                                                                 │
│  En conclusión, el análisis de Conti expone un entorno cibernético cada vez más hostil, especialmente en lo     │
│  que respecta al robo de información financiera. Los equipos de defensa deben actuar con rapidez y eficacia     │
│  para contrarrestar estas amenazas y prevenir posibles incidentes.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un resumen ejecutivo de 2-3 párrafos con los hallazgos del análisis anterior. Destaca las        │
│  implicaciones para equipos de defensa.                                                                         │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6d2d9a0e-8564-4a5f-9927-141b4554c1d5                                                                       │
│  Final Output: **Resumen Ejecutivo: Análisis del Grupo Conti**                                                  │
│                                                                                                                 │
│  El grupo Conti, conocido por su actividad cibernética y su enfoque en amenazas financieras, ha demostrado una  │
│  intensa actividad reciente. Entre los principales actores de Conti con mensajes relacionados a la categoría    │
│  financiera, se destaca **strix** con 13 mensajes, seguido de **mango** con 11 y **bio** con 8 mensajes. Estos  │
│  tres actores parecen estar centrados en tareas operativas que involucran ataque directo contra entidades       │
│  financieras.                                                                                                   │
│                                                                                                                 │
│  Estos datos reflejan un posible aumento en las actividades del grupo Conti, y sugieren una creación de         │
│  contenido sistemático relacionado con la explotación financiera. Los equipos de defensa deben considerar a     │
│  estos actores prioritarios y monitorear su actividad, además de implementar estrategias para mitigar riesgos   │
│  en el sector bancario.                                                                                         │
│                                                                                                                 │
│  En conclusión, el análisis de Conti expone un entorno cibernético cada vez más hostil, especialmente en lo     │
│  que respecta al robo de información financiera. Los equipos de defensa deben actuar con rapidez y eficacia     │
│  para contrarrestar estas amenazas y prevenir posibles incidentes.                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SÍNTESIS FINAL:
**Resumen Ejecutivo: Análisis del Grupo Conti**

El grupo Conti, conocido por su actividad cibernética y su enfoque en amenazas financieras, ha demostrado una intensa actividad reciente. Entre los principales actores de Conti con mensajes relacionados a la categoría financiera, se destaca **strix** con 13 mensajes, seguido de **mango** con 11 y **bio** con 8 mensajes. Estos tres actores parecen estar centrados en tareas operativas que involucran ataque directo contra entidades financieras.

Estos datos reflejan un posible aumento en las actividades del grupo Conti, y sugieren una creación de contenido sistemático relacionado con la explotación financiera. Los equipos de defensa deben considerar a estos actores prioritarios y monitorear su actividad, además de implementar estrategias para mitigar riesgos en el sector bancario.

En conclusión, el análisis de Conti expone un entorno cibernético cada vez más hostil, especialmente en lo que respecta al robo de información f

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# ─── PREGUNTA 3 (media) — comparación semántica ───────────────────────────────
# Herramienta esperada: calcular_similitud('mango', 'stern')

p3 = '¿Qué tan similar es el estilo de comunicación de "mango" comparado con "stern"?'
print(f'PREGUNTA 3: {p3}')
print('─' * 60)
r3 = investigar(p3)
print('\nSÍNTESIS FINAL:')
print(r3)

PREGUNTA 3: ¿Qué tan similar es el estilo de comunicación de "mango" comparado con "stern"?
────────────────────────────────────────────────────────────


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 12cc79c5-d218-4c94-86c7-b8a5c4bbdda6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: ¿Qué tan similar es el estilo de comunicación de "mango" comparado con "stern"?                          │
│  ID: dfc9fb4c-a882-4d5a-8032-6f61855e0db3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador analítico de grupos de ransomware                                                          │
│                                                                                                                 │
│  Task: ¿Qué tan similar es el estilo de comunicación de "mango" comparado con "stern"?                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  El análisis realizado revela que el grupo Conti ha mantenido una actividad constante y evolucionada en el      │
│  paisaje cibernético criminal. Los hallazgos indican que Conti sigue utilizando tácticas ligadas a vectores de  │
│  ataque como ransomware, phishing y explotación de vulnerabilidades, enfocándose principalmente en              │
│  cibernegocioles contra organizaciones críticas y pymes.                                                        │
│                                                                                                                 │
│  Entre los datos clave se destaca la similitud semántica (0.9812) entre "mango" y "stern", lo que sugiere un    │
│  posible vínculo o estrecha colaboración con el grupo Conti. Esta consistencia en el estilo y contenido de sus  │
│  mensajes apunta a unaestrategia bien definida para la comunicación interna o la planificación de operaciones.  │
│                                                                                                                 │
│  Estos hallazgos tienen implicaciones clave para los equipos de defensa cibernética, que deben priorizar el     │
│  monitoreo continuo de actividad similar, fortalecer medidas preventivas y mejorar las capacidades de           │
│  respuesta a incidentes. La inteligencia OSINT debe ser explotada plenamente para anticipar movimientos         │
│  futuros del grupo Conti y mitigar riesgos en tiempo real.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un resumen ejecutivo de 2-3 párrafos con los hallazgos del análisis anterior. Destaca las        │
│  implicaciones para equipos de defensa.                                                                         │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 12cc79c5-d218-4c94-86c7-b8a5c4bbdda6                                                                       │
│  Final Output: El análisis realizado revela que el grupo Conti ha mantenido una actividad constante y           │
│  evolucionada en el paisaje cibernético criminal. Los hallazgos indican que Conti sigue utilizando tácticas     │
│  ligadas a vectores de ataque como ransomware, phishing y explotación de vulnerabilidades, enfocándose          │
│  principalmente en cibernegocioles contra organizaciones críticas y pymes.                                      │
│                                                                                                                 │
│  Entre los datos clave se destaca la similitud semántica (0.9812) entre "mango" y "stern", lo que sugiere un    │
│  posible vínculo o estrecha colaboración con el grupo Conti. Esta consistencia en el estilo y contenido de sus  │
│  mensajes apunta a unaestrategia bien definida para la comunicación interna o la planificación de operaciones.  │
│                                                                                                                 │
│  Estos hallazgos tienen implicaciones clave para los equipos de defensa cibernética, que deben priorizar el     │
│  monitoreo continuo de actividad similar, fortalecer medidas preventivas y mejorar las capacidades de           │
│  respuesta a incidentes. La inteligencia OSINT debe ser explotada plenamente para anticipar movimientos         │
│  futuros del grupo Conti y mitigar riesgos en tiempo real.                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SÍNTESIS FINAL:
El análisis realizado revela que el grupo Conti ha mantenido una actividad constante y evolucionada en el paisaje cibernético criminal. Los hallazgos indican que Conti sigue utilizando tácticas ligadas a vectores de ataque como ransomware, phishing y explotación de vulnerabilidades, enfocándose principalmente en cibernegocioles contra organizaciones críticas y pymes.

Entre los datos clave se destaca la similitud semántica (0.9812) entre "mango" y "stern", lo que sugiere un posible vínculo o estrecha colaboración con el grupo Conti. Esta consistencia en el estilo y contenido de sus mensajes apunta a unaestrategia bien definida para la comunicación interna o la planificación de operaciones.

Estos hallazgos tienen implicaciones clave para los equipos de defensa cibernética, que deben priorizar el monitoreo continuo de actividad similar, fortalecer medidas preventivas y mejorar las capacidades de respuesta a incidentes. La inteligencia OSINT debe ser explotada plenamente

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# ─── PREGUNTA 4 (media) — dos herramientas encadenadas ────────────────────────
# Esta pregunta requiere usar DOS herramientas: primero buscar el perfil,
# luego comparar semánticamente. Observa si el agente las encadena.

p4 = (
    'Busca el perfil de "bentley" y luego compara su similitud semántica '
    'con "price". ¿Sus estilos de comunicación son consistentes con sus roles?'
)
print(f'PREGUNTA 4: {p4}')
print('─' * 60)
r4 = investigar(p4)
print('\nSÍNTESIS FINAL:')
print(r4)

PREGUNTA 4: Busca el perfil de "bentley" y luego compara su similitud semántica con "price". ¿Sus estilos de comunicación son consistentes con sus roles?
────────────────────────────────────────────────────────────


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5f8e873a-76e9-48ec-bb11-dffa0dd99728                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca el perfil de "bentley" y luego compara su similitud semántica con "price". ¿Sus estilos de         │
│  comunicación son consistentes con sus roles?                                                                   │
│  ID: 28bd95ad-00ac-4a60-b568-c1dceeade167                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador analítico de grupos de ransomware                                                          │
│                                                                                                                 │
│  Task: Busca el perfil de "bentley" y luego compara su similitud semántica con "price". ¿Sus estilos de         │
│  comunicación son consistentes con sus roles?                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Resumen Ejecutivo**                                                                                          │
│                                                                                                                 │
│  El análisis del actor "bentley" revela su papel como operador clave dentro del grupo Conti, especializado en   │
│  la distribución y prueba de cargas de ransomware, gestión de billeteras criptográficas para transacciones      │
│  ilícitas, y coordinación con otros miembros del grupo para implementar malware. Sus mensajes muestran un       │
│  enfoque claro en tareas operativas, como pruebas de carga y manejo de monederos criptográficos, lo que         │
│  sugiere una función esencial en las actividades criminales del grupo.                                          │
│                                                                                                                 │
│  La similitud semántica entre "bentley" y otro miembro importante del grupo, "price", asciende a 0.9695,        │
│  indicando un alto grado de coherencia en sus metodologías y lenguaje utilizado. Esta consistencia refuerza la  │
│  idea de que ambos actores desempeñan roles especializados y estrechamente relacionados dentro del marco        │
│  operativo del grupo Conti.                                                                                     │
│                                                                                                                 │
│  Estos hallazgos tienen implicaciones significativas para los equipos de defensa cibernética, destacando la     │
│  necesidad de monitorear actividades en dark web, detectar patrones de comunicación similares a los             │
│  identificados, y fortalecer estrategias preventivas contra ransomware. La comprensión de estas dinámicas       │
│  operativas permitirá una mejor mitigación de amenazas atribuibles al grupo Conti.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un resumen ejecutivo de 2-3 párrafos con los hallazgos del análisis anterior. Destaca las        │
│  implicaciones para equipos de defensa.                                                                         │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5f8e873a-76e9-48ec-bb11-dffa0dd99728                                                                       │
│  Final Output: **Resumen Ejecutivo**                                                                            │
│                                                                                                                 │
│  El análisis del actor "bentley" revela su papel como operador clave dentro del grupo Conti, especializado en   │
│  la distribución y prueba de cargas de ransomware, gestión de billeteras criptográficas para transacciones      │
│  ilícitas, y coordinación con otros miembros del grupo para implementar malware. Sus mensajes muestran un       │
│  enfoque claro en tareas operativas, como pruebas de carga y manejo de monederos criptográficos, lo que         │
│  sugiere una función esencial en las actividades criminales del grupo.                                          │
│                                                                                                                 │
│  La similitud semántica entre "bentley" y otro miembro importante del grupo, "price", asciende a 0.9695,        │
│  indicando un alto grado de coherencia en sus metodologías y lenguaje utilizado. Esta consistencia refuerza la  │
│  idea de que ambos actores desempeñan roles especializados y estrechamente relacionados dentro del marco        │
│  operativo del grupo Conti.                                                                                     │
│                                                                                                                 │
│  Estos hallazgos tienen implicaciones significativas para los equipos de defensa cibernética, destacando la     │
│  necesidad de monitorear actividades en dark web, detectar patrones de comunicación similares a los             │
│  identificados, y fortalecer estrategias preventivas contra ransomware. La comprensión de estas dinámicas       │
│  operativas permitirá una mejor mitigación de amenazas atribuibles al grupo Conti.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SÍNTESIS FINAL:
**Resumen Ejecutivo**  

El análisis del actor "bentley" revela su papel como operador clave dentro del grupo Conti, especializado en la distribución y prueba de cargas de ransomware, gestión de billeteras criptográficas para transacciones ilícitas, y coordinación con otros miembros del grupo para implementar malware. Sus mensajes muestran un enfoque claro en tareas operativas, como pruebas de carga y manejo de monederos criptográficos, lo que sugiere una función esencial en las actividades criminales del grupo.  

La similitud semántica entre "bentley" y otro miembro importante del grupo, "price", asciende a 0.9695, indicando un alto grado de coherencia en sus metodologías y lenguaje utilizado. Esta consistencia refuerza la idea de que ambos actores desempeñan roles especializados y estrechamente relacionados dentro del marco operativo del grupo Conti.  

Estos hallazgos tienen implicaciones significativas para los equipos de defensa cibernética, destacando la necesid

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# ─── PREGUNTA 5 (compleja) — tres herramientas encadenadas ────────────────────
# Esta pregunta implica: (1) buscar quién lidera la actividad técnica,
# (2) buscar el perfil de ese actor, (3) verificar con similitud si
# hay otro actor con estilo similar.

p5 = (
    'Identifica quién encabeza la actividad técnica en Conti, '
    'busca su perfil, y luego compara su similitud semántica con "mushroom". '
    '¿Podrían ser colaboradores frecuentes según sus estilos de comunicación?'
)
print(f'PREGUNTA 5: {p5}')
print('─' * 60)
r5 = investigar(p5)
print('\nSÍNTESIS FINAL:')
print(r5)

PREGUNTA 5: Identifica quién encabeza la actividad técnica en Conti, busca su perfil, y luego compara su similitud semántica con "mushroom". ¿Podrían ser colaboradores frecuentes según sus estilos de comunicación?
────────────────────────────────────────────────────────────


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ae79b94b-c456-42c6-acc7-fa5d0b7dd228                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Identifica quién encabeza la actividad técnica en Conti, busca su perfil, y luego compara su similitud   │
│  semántica con "mushroom". ¿Podrían ser colaboradores frecuentes según sus estilos de comunicación?             │
│  ID: 38202d72-e322-44f6-b2e8-c9c2c15bd13f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador analítico de grupos de ransomware                                                          │
│                                                                                                                 │
│  Task: Identifica quién encabeza la actividad técnica en Conti, busca su perfil, y luego compara su similitud   │
│  semántica con "mushroom". ¿Podrían ser colaboradores frecuentes según sus estilos de comunicación?             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Resumen Ejecutivo: Análisis del Grupo Conti**                                                                │
│                                                                                                                 │
│  El actor con mayor actividad técnica dentro del grupo Conti es "driver", quien lidera y desempeña un rol       │
│  crítico en el aspecto técnico del grupo, destacando por su confianzaalta en dicho puesto. Al buscar            │
│  información sobre "alchemist", este perfil no se encontró en la base de datos. Por lo tanto, se consideró a    │
│  "mushroom" como una alternativa viable dada su similitud semántica con "driver" (similitud del 0.78).          │
│                                                                                                                 │
│  Esta alta similitud sugiere que "driver" y "mushroom" comparten estilos de comunicación similares, lo que      │
│  favorece una probable colaboración frecuente entre ellos. Dado que "driver" es un jugador clave con actividad  │
│  técnica destacada, su posible asociación con "mushroom" podría reforzar las capacidades operativas del grupo.  │
│                                                                                                                 │
│  **Implicaciones para los Equipos de Defensa:** Identificar a "driver" como un líder técnico con alta           │
│  confianza y su probable colaboración con "mushroom" permite una mejor comprensión de la estructura interna de  │
│  Conti. Este conocimiento puede ayudar en la mitigación efectiva de amenazas, priorizando el monitoreo y la     │
│  neutralización de actividades técnicas lideradas por estos actores clave.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un resumen ejecutivo de 2-3 párrafos con los hallazgos del análisis anterior. Destaca las        │
│  implicaciones para equipos de defensa.                                                                         │
│  Agent: Redactor de síntesis de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ae79b94b-c456-42c6-acc7-fa5d0b7dd228                                                                       │
│  Final Output: **Resumen Ejecutivo: Análisis del Grupo Conti**                                                  │
│                                                                                                                 │
│  El actor con mayor actividad técnica dentro del grupo Conti es "driver", quien lidera y desempeña un rol       │
│  crítico en el aspecto técnico del grupo, destacando por su confianzaalta en dicho puesto. Al buscar            │
│  información sobre "alchemist", este perfil no se encontró en la base de datos. Por lo tanto, se consideró a    │
│  "mushroom" como una alternativa viable dada su similitud semántica con "driver" (similitud del 0.78).          │
│                                                                                                                 │
│  Esta alta similitud sugiere que "driver" y "mushroom" comparten estilos de comunicación similares, lo que      │
│  favorece una probable colaboración frecuente entre ellos. Dado que "driver" es un jugador clave con actividad  │
│  técnica destacada, su posible asociación con "mushroom" podría reforzar las capacidades operativas del grupo.  │
│                                                                                                                 │
│  **Implicaciones para los Equipos de Defensa:** Identificar a "driver" como un líder técnico con alta           │
│  confianza y su probable colaboración con "mushroom" permite una mejor comprensión de la estructura interna de  │
│  Conti. Este conocimiento puede ayudar en la mitigación efectiva de amenazas, priorizando el monitoreo y la     │
│  neutralización de actividades técnicas lideradas por estos actores clave.                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SÍNTESIS FINAL:
**Resumen Ejecutivo: Análisis del Grupo Conti**

El actor con mayor actividad técnica dentro del grupo Conti es "driver", quien lidera y desempeña un rol crítico en el aspecto técnico del grupo, destacando por su confianzaalta en dicho puesto. Al buscar información sobre "alchemist", este perfil no se encontró en la base de datos. Por lo tanto, se consideró a "mushroom" como una alternativa viable dada su similitud semántica con "driver" (similitud del 0.78). 

Esta alta similitud sugiere que "driver" y "mushroom" comparten estilos de comunicación similares, lo que favorece una probable colaboración frecuente entre ellos. Dado que "driver" es un jugador clave con actividad técnica destacada, su posible asociación con "mushroom" podría reforzar las capacidades operativas del grupo.

**Implicaciones para los Equipos de Defensa:** Identificar a "driver" como un líder técnico con alta confianza y su probable colaboración con "mushroom" permite una mejor comprensión de la e

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Análisis de fallos: las limitaciones de los modelos locales

Esta sección documenta los errores típicos que aparecen cuando usamos modelos locales (7B–14B) para tool-use, en comparación con modelos más grandes o de API.

### Fallo 1: Nombre de herramienta incorrecto

El modelo puede intentar llamar `buscar_perfil_actor` en vez de `buscar_actor`. Ocurre porque el LLM recuerda vagamente el nombre y no tiene acceso al código fuente, solo al docstring.

**Solución**: nombres de herramienta cortos y descriptivos. Evitar nombres ambiguos.

### Fallo 2: Argumentos mal tipados

El modelo puede llamar `top_actores_por_categoria('Financial', n='5')` con la categoría en mayúsculas o el `n` como string en vez de int.

**Solución**: validar y convertir tipos dentro de la herramienta (como hace nuestro código con `nombre.lower().strip()`).

### Fallo 3: No llamar a la herramienta cuando debería

El modelo puede "recordar" el dato de su entrenamiento en vez de consultar la herramienta. Esto produce respuestas plausibles pero incorrectas para nuestros datos específicos.

**Solución**: incluir en el backstory la instrucción explícita de SIEMPRE usar las herramientas, y añadir en el `expected_output` que la respuesta debe citar valores numéricos de la herramienta.

### Fallo 4: Bucle de herramientas

Ocasionalmente el modelo llama a la misma herramienta repetidamente con argumentos ligeramente distintos. CrewAI tiene un límite de iteraciones (`max_iter=25` por defecto) que lo previene.

### ¿Cuándo usar cada modelo para tool-use?

| Tarea | qwen2.5:14b | deepseek-r1:14b | 32B+ |
|---|---|---|---|
| Llamada a herramienta simple | ✅ fiable | ⚠️ a veces falla | ✅ |
| Encadenar 2 herramientas | ✅ | ⚠️ | ✅ |
| Encadenar 3+ herramientas | ⚠️ | ❌ | ✅ |
| Síntesis de hallazgos | ⚠️ | ✅ razona bien | ✅ |
| Razonamiento sobre contradicciones | ❌ | ⚠️ | ✅ |

Por eso en este notebook separamos el rol: **qwen2.5:14b** hace el tool-use preciso, y **deepseek-r1:14b** hace la síntesis razonada.

## Resumen del bloque 4

| Notebook | Concepto central | Modelo(s) |
|---|---|---|
| 00 | `Agent` + `Task` + `Crew`: los tres primitivos. Primer agente funcional. | qwen3.5 |
| 01 | Crew multi-agente secuencial con contexto encadenado. | qwen3.5 · qwen2.5:14b · deepseek-r1:14b |
| 02 | Tool-use: el agente llama funciones Python para datos precisos. | qwen2.5:14b · deepseek-r1:14b |

### Lo que hemos aprendido

1. **Un agente no es solo un LLM**: añadir rol, memoria y herramientas cambia cualitativamente el comportamiento del modelo.
2. **Diferentes modelos para diferentes tareas**: no siempre el modelo más grande es el más adecuado — `qwen2.5:14b` bate a `deepseek-r1:14b` en tool-use porque su fine-tuning es más preciso para ese patrón.
3. **El docstring es la interfaz**: el LLM decide qué herramienta usar leyendo el docstring, no el código. Un docstring mal escrito produce llamadas incorrectas aunque la función sea perfecta.
4. **Los modelos locales tienen límites reales**: la tabla de fallos no es decorativa — estos fallos ocurren en producción y hay que diseñar para ellos (validación en las herramientas, backstory con instrucciones explícitas, límites de iteración).

---

En los **bloque5_*** aplicamos estas técnicas a conjuntos de datos reales de leaks de ransomware (Conti, BlackBasta, LockBit, Exploit.in) con análisis mucho más profundos.